In [17]:
import torch
from torch.nn import functional as F
import tiktoken
from train_gpt_124M import GPTConfig, GPT

device = 'cpu'
checkpoint = torch.load('weights/model_19072_fineweb_124M.pt', weights_only=False, map_location=torch.device(device))
config = checkpoint['config']
model = GPT(config)
model.load_state_dict(checkpoint['model'])
model = model.to(device)
model.eval()
enc = tiktoken.get_encoding('gpt2')

In [18]:
sum(p.numel() for p in model.parameters())

124475904

In [10]:
def generate(inp, num_return_sequences=4, max_tokens = 128):
    tokens = enc.encode(inp) # (T,)
    tokens = torch.tensor(tokens, dtype=torch.long)
    tokens = tokens.unsqueeze(0).repeat(num_return_sequences, 1)
    xgen = tokens.to(device) # (B, T)
    
    for _ in range(max_tokens):
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            logits, loss = model(xgen) # (B, T, C)
        logits = logits[:, -1, :] # (B, C)
        probs = F.softmax(logits, dim=-1)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1) # (B, 50)
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        xcol = torch.gather(topk_indices, -1, ix)
        xgen = torch.cat((xgen, xcol), dim=1)

    for i in range(num_return_sequences):
        tokens = xgen[i, :max_tokens].tolist()
        decoded = enc.decode(tokens)
        print(decoded)
        print('##############################')


generate('The capital of France is')

The capital of France is not in
The French constitution declares that the French people enjoy a freedom of religion. The French citizens who speak to themselves as Protestants do not necessarily practice any religion.
The French flag was first adopted in 1870. It depicts a green field with 13 stars representing the 13 colonies.
The flag has since become a symbol of national unity. The current flag design was first introduced in 1761. The colors of the flag are composed of seven horizontal stripes that are blue and red surrounded by violet white stripes and white red stripes.
The French flag has the motto “Lettres est le commune”
##############################
The capital of France is the Place de la Concorde, the only place of great and grandeur among its great cities to be laid waste. There were many grand houses, a few simple gardens, a town on the edge of the river, and plenty of timber. In general there were gardens with more than 250 plants and trees which were very rich. The hous

In [15]:
from train_gpt import GPTConfig, GPT

checkpoint = torch.load('weights/model_10699_climbmix_700M.pt', weights_only=False, map_location=torch.device(device))
config = checkpoint['config']
model = GPT(config)

state_dict = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model'].items()}

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 1536)
    (wpe): Embedding(2048, 1536)
    (h): ModuleList(
      (0-23): 24 x Block(
        (ln_1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=1536, out_features=4608, bias=True)
          (c_proj): Linear(in_features=1536, out_features=1536, bias=True)
        )
        (ln_2): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=1536, out_features=6144, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=6144, out_features=1536, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1536, out_features=50304, bias=False)
)

In [16]:
sum(p.numel() for p in model.parameters())

760372224

In [ ]:
generate('The capital of France is')